# Лекција 11 - Протокол агент-а-агенту (A2A)


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Шта је А2А протокол?

**Agent-to-Agent (A2A) протокол** је отворени стандард који омогућава AI агентима да комуницирају,
откривају једни друге и сарађују — чак и када су изграђени на различитим оквирима или смештени
код различитих услуга.

Кључни појмови:

- **Откривање** – Агенти објављују *Agent Card* који описује њихове способности, чинећи
  лакшим за друге агенте (или оркестраторе) да пронађу правог стручњака за задатак.
- **Пренос порука** – Агенти размењују структуриране поруке кроз заједнички протокол, тако да
  захтев од једног агента може бити разумљив и извршен од стране другог без обзира на његову
  унутрашњу имплементацију.
- **Животни циклус задатка** – А2А дефинише стања као што су *poslato*, *ради се*, *завршено* и
  *није успело*, дајући оркестратору потпун увид у то како се одређени задатак извршава.

У овој лекцији симулирамо сарадњу у стилу А2А повезивањем три специјализована агента за путовања
у радни ток где сваки агент даје свој допринос и прослеђује резултате следећем.


## Креирање специјализованих туристичких агената


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Вишеагентска сарадња путем радног тока

Повезујемо три агента у секвенцијални радни ток који одсликава прослеђивање порука A2A:

1. **CurrencyExchangeAgent** прими кориснички захтев и пружи смернице о валути.
2. **ActivityPlannerAgent** прими обогаћени контекст и дода препоруке за активности.
3. **TravelManagerAgent** синтетише оба уноса у коначни путни извештај.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Разумевање A2A у продукцији

У продукционом окружењу A2A протокол откључава моћне сценарије преко сервиса:

| Могућност | Опис |
|---|---|
| **Интероперабилност између различитих фрејмворка** | Агенти изграђени у једном фрејмворку могу делегирати задатке агентима изграђеним у било ком другом A2A-компатибилном фрејмворку, омогућавајући праву интероперабилност између организација. |
| **Границе сервиса** | Агенти могу бити смештени у посебним микросервисима, облачним регионима, или чак различитим организацијама, а ипак сарађивати беспрекорно. |
| **Динамичко откривање** | Организатор може у реалном времену упитати регистар Agent Card картица да пронађе најбоље одабраног специјалисту за дати подсабатак. |
| **Стримовање и push обавештења** | A2A подржава Server-Sent Events (SSE) за ажурирања напретка у реалном времену и push обавештења за дуготрајне задатке. |

Радни ток који смо горе направили је поједностављена, у процесу верзија овог обрасца. У стварној
имплементацији сваки агент би излагао HTTP ендпоинт, објављивао Agent Card и комуницирао
преко A2A JSON-RPC протокола.


## Резиме

У овој лекцији сте научили:

1. **Шта је А2А протокол** — отворени стандард за откривање, слање порука и управљање задацима између агената.
   и управљање задацима.
2. **Како створити специјализоване агенте** — агента за размену валута, агента за планирање активности,
   и оркестратор за управљање путовањима.
3. **Како повезати агенте у ток рада** — коришћењем `WorkflowBuilder` за моделирање секвенцијалног
   преноса порука између агената.
4. **Како А2А функционише у продукцији** — омогућавање сарадње између различитих оквира и сервиса
   уз динамичко откривање и ажурирања у реалном времену.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
